In [2]:
import re
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import torch.nn.functional as F
import torch.optim as optim
import time
import random
import matplotlib.pyplot as plt
from tqdm import tqdm

In [3]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'当前设备:{device}')

SOS_token = 0
EOS_token = 1
MAX_LENGTH = 10
data_path = 'eng-fra-v2.txt'

当前设备:cuda


In [4]:
def normalizeString(s):
    
    s = s.lower().strip()
    
    s = re.sub(r'([.!?])', r' \1', s)
    
    s = re.sub(r'[^a-zA-Z.!?]+', r' ', s)
    
    return s

In [5]:
def get_data():
    with open(data_path, 'r', encoding='utf-8') as src_f:
        lines = src_f.readlines()

        word_pairs = [[normalizeString(s) for s in line.split('\t')] for line in lines]

        eng_word2index = {'SOS': 0, 'EOS': 1}
        eng_word_n = 2
        french_word2index = {'SOS': 0, 'EOS': 1}
        french_word_n = 2

        for pair in word_pairs:
            for eng_word in pair[0].split(' '):
                if eng_word not in eng_word2index:
                    eng_word2index[eng_word] = eng_word_n
                    eng_word_n += 1
            for french_word in pair[1].split(' '):
                if french_word not in french_word2index:
                    french_word2index[french_word] = french_word_n
                    french_word_n += 1

        eng_index2word = {v: k for k, v in eng_word2index.items()}
        french_index2word = {v: k for k, v in french_word2index.items()}

        print(f'英语词汇表大小: {eng_word_n}')
        print(f'法语词汇表大小: {french_word_n}')

        return eng_word2index, eng_index2word, eng_word_n, french_word2index, french_index2word, french_word_n, word_pairs

In [6]:
eng_word2index, eng_index2word, eng_word_n, french_word2index, french_index2word, french_word_n, word_pairs = get_data()

英语词汇表大小: 2803
法语词汇表大小: 4345


In [7]:
class SeqDataset(Dataset):
    def __init__(self, word_pairs):
        self.word_pairs = word_pairs
        self.sample_len = len(word_pairs)

    def __len__(self):
        return self.sample_len

    def __getitem__(self, index):
        index = min(max(index, 0), self.sample_len - 1)

        x = self.word_pairs[index][0]
        y = self.word_pairs[index][1]

        x = [eng_word2index[word] for word in x.split(' ')]
        x.append(EOS_token)

        y = [french_word2index[word] for word in y.split(' ')]
        y.append(EOS_token)

        tensor_x = torch.tensor(x, dtype=torch.long, device=device)
        tensor_y = torch.tensor(y, dtype=torch.long, device=device)

        return tensor_x, tensor_y

In [8]:
def get_dataloader():
    dataset = SeqDataset(word_pairs)
    dataloader = DataLoader(dataset, batch_size=1, shuffle=True)

    return dataloader

In [ ]:
num_layers = 1
hidden_size = 256
dropout_p = 0.1
batch_size = 1

# RNN
# 输入形状：    [seq_len, batch_size, input_size]                           句子长度, 批次大小, 词嵌入维度
# 输入隐藏状态： [num_layers * num_directions, batch_size, hidden_size]     RNN层数 * 方向数, 批次大小, 隐藏层维度
# 输出形状：    [seq_len, batch_size, hidden_size * num_directions]         句子长度, 批次大小, 隐藏层维度
# 输出隐藏状态： [num_layers * num_directions, batch_size, hidden_size]     RNN层数 * 方向数, 批次大小, 隐藏层维度

In [ ]:
class EncoderRNN(nn.Module):
    def __init__(self, input_size, hidden_size):
        super().__init__()
        self.input_size = input_size
        self.hidden_size = hidden_size

        self.embedding = nn.Embedding(input_size, hidden_size)

        self.gru = nn.GRU(hidden_size, hidden_size, num_layers)

    def forward(self, input, hidden):
        """
        input [seq_len, batch_size]
        hidden [num_layers, batch_size, hidden_size]
        """
        output = self.embedding(input) # [seq_len, batch_size, hidden_size]

        output, hidden = self.gru(output, hidden)

        return output, hidden

    def init_hidden(self):
        return torch.zeros(num_layers, batch_size, hidden_size, device=device)

In [ ]:
"""
decoder流程：
    由输入词和encoder最后的隐藏状态(作为初始隐藏状态)计算出注意力权重，
    再将注意力权重应用在encoder的所有输出上，得到中间语义张量C，
    将C作为输入上一次隐藏状态作为隐藏状态经过GRU获得输出，
    再将输出转为词的概率

forward参数:
    input: 输入词, 形状为: [seq_len, batch_size]
    hidden: 初始隐藏状态, 形状为: [num_layers * num_directions, batch_size, hidden_size]
    encoder_outputs: 编码器所有输出(中间语义张量C, 计算出注意力权重后, 作为GRU输入), 形状为: [seq_len, batch_size, hidden_size]
"""

class AttnDecoderRNN(nn.Module):
    def __init__(self, output_size, hidden_size, dropout_p=0.1, max_length=MAX_LENGTH):
        super().__init__()
        self.output_size = output_size
        self.hidden_size = hidden_size
        self.dropout_p = dropout_p
        self.max_length = max_length

        self.embedding = nn.Embedding(output_size, hidden_size)

        self.attn = nn.Linear(hidden_size * 2, max_length)

        self.attn_combine = nn.Linear(hidden_size * 2, hidden_size)

        self.dropout = nn.Dropout(dropout_p)

        self.gru = nn.GRU(hidden_size, hidden_size, num_layers)

        self.out = nn.Linear(hidden_size, output_size)

        self.softmax = nn.LogSoftmax(dim=-1)

    def forward(self, input, hidden, encoder_outputs):
        """
        input [seq_len, batch_size]
        hidden [num_layers, batch_size, hidden_size]
        encoder_outputs [max_length, hidden_size]
        """
        embedded = self.embedding(input) # [seq_len, batch_size, hidden_size]
        embedded = self.dropout(embedded)

        # [seq_len, batch_size, hidden_size] -> [batch_size, hidden_size * 2] -> [batch_size, max_length]
        attn_weights = F.softmax(self.attn(torch.cat((embedded[0], hidden[0]), 1)), dim=1)

        # [1, batch_size, hidden_size]
        attn_applied = torch.bmm(attn_weights.unsqueeze(0), encoder_outputs.unsqueeze(0))

        output = self.attn_combine(torch.cat((embedded[0], attn_applied[0]), 1)).unsqueeze(0) # [1, 1, hidden_size]

        output = F.relu(output)

        output, hidden = self.gru(output, hidden)

        output = self.softmax(self.out(output[0]))

        return output, hidden, attn_weights

    def init_hidden(self):
        return torch.zeros(num_layers, batch_size, hidden_size, device=device)

In [12]:
my_lr, epochs, teacher_forcing_ratio, print_interval_num, plot_interval_num = 1e-4, 50, 0.5, 1000, 100

In [ ]:
def train_iters(x, y, my_encoder, my_decoder, my_adam_encode, my_adam_decode, my_crossentropy_loss):
    encoder_hidden = my_encoder.init_hidden()
    encoder_output, encoder_hidden = my_encoder(x, encoder_hidden)

    encoder_output_c = torch.zeros(MAX_LENGTH, my_encoder.hidden_size, device=device)
    for idx in range(x.shape[1]):
        encoder_output_c[idx] = encoder_output[idx, 0]
    
    decoder_hidden = encoder_hidden

    decoder_input = torch.tensor([[SOS_token]], device=device)

    loss, y_len = 0.0, y.shape[1]

    use_teacher_forcing = True if random.random() < teacher_forcing_ratio else False
    if(use_teacher_forcing):
        for i in range(y_len):
            output_y, decoder_hidden, attn_weights = my_decoder(decoder_input, decoder_hidden, encoder_output_c)

            target_y = y[0][i].view(1, -1)

            loss += my_crossentropy_loss(output_y, target_y)

            decoder_input = y[0][i].view(1, -1)

    else:
        for i in range(y_len):
            output_y, decoder_hidden, attn_weights = my_decoder(decoder_input, decoder_hidden, encoder_output_c)

            target_y = y[0][i].view(1)

            loss += my_crossentropy_loss(output_y, target_y)

            topv, topi = output_y.topk(1)

            if(topi.squeeze().item() == EOS_token):
                break

            decoder_input = topi.detach()

    my_adam_encode.zero_grad()
    my_adam_decode.zero_grad()

    loss.backward()

    my_adam_encode.step()
    my_adam_decode.step()

    return loss.item() / y_len

a: tensor([[[3, 1, 3, 4],
         [2, 1, 2, 3],
         [2, 4, 2, 2]],

        [[1, 4, 3, 2],
         [1, 1, 4, 4],
         [3, 4, 3, 1]]])
a[:, 0, :]: tensor([1, 4])
a.shape: torch.Size([2])
